# Sprint 6  :  Test End-to-End IoT (Mosquitto)

**AquaSense AI**

Objectifs du Sprint :
- Démarrer le broker Mosquitto en local.
- Lancer le simulateur de 50 pompes (saine, dégradation, failure).
- Lancer le consumer MQTT pour les prédictions en temps réel.
- Vérifier la base SQLite et la latence.

Note : **IMPORTANT : Ce notebook doit être exécuté en LOCAL (pas sur Colab), car il nécessite un broker Mosquitto local fonctionnel sur le port 1883.**

## 1. Démarrer le Broker Mosquitto

Si vous avez Docker d'installé, vous pouvez lancer Mosquitto avec cette commande dans un terminal séparé :
```bash
docker run -it -p 1883:1883 eclipse-mosquitto
```
Si Mosquitto est installé nativement, assurez-vous que le service tourne (`sudo systemctl start mosquitto` ou lancement via Windows Services).

In [1]:
import subprocess
import sys
import time
import sqlite3
from pathlib import Path

PROJECT_ROOT = Path("..").resolve()
DB_PATH = PROJECT_ROOT / "data" / "mqtt" / "aquasense.db"
PYTHON = sys.executable


## 2 & 3. Lancer Simulator et Consumer en arrière-plan

Nous allons utiliser Python `subprocess` pour lancer les deux scripts de manière non-bloquante.

In [2]:
print("Demarrage consumer MQTT...")
cons_proc = subprocess.Popen(
    [PYTHON, "-m", "src.mqtt_consumer"],
    cwd=str(PROJECT_ROOT),
)
print("Demarrage simulateur (5 pompes, 10 s)...")
sim_proc = subprocess.Popen(
    [PYTHON, "-m", "src.simulator", "--pumps", "5", "--interval", "2"],
    cwd=str(PROJECT_ROOT),
)
time.sleep(12)
print("Pause terminee. Arret des processus...")
sim_proc.terminate()
cons_proc.terminate()


Demarrage consumer MQTT...
Demarrage simulateur (5 pompes, 10 s)...


Pause terminee. Arret des processus...


## 4. Vérifier les données insérées en base

In [3]:
if not DB_PATH.exists():
    print(f"Base absente : {DB_PATH}")
else:
    conn = sqlite3.connect(DB_PATH)
    t = conn.execute("SELECT COUNT(*) FROM telemetry").fetchone()[0]
    p = conn.execute("SELECT COUNT(*) FROM predictions").fetchone()[0]
    print(f"telemetry={t}, predictions={p}")
    conn.close()


telemetry=113474, predictions=113472


## 5 & 6. Tester les 3 scénarios (Saine, Dégradation, Failure)
On observe comment la prédiction et les données capteurs s'affichent.

In [4]:
import pandas as pd

if not DB_PATH.exists():
    print("Pas de donnees.")
else:
    conn = sqlite3.connect(DB_PATH)
    df_telemetry = pd.read_sql_query(
        "SELECT pump_id, timestamp, pressure, vibration, flow FROM telemetry "
        "ORDER BY timestamp DESC LIMIT 50",
        conn,
    )
    preds = pd.read_sql_query(
        "SELECT pump_id, timestamp, prediction FROM predictions "
        "ORDER BY timestamp DESC LIMIT 20",
        conn,
    )
    conn.close()
    print("Dernieres telemetries :")
    print(df_telemetry.head(5).to_string(index=False))
    print("\nDernieres predictions :")
    print(preds.head(5).to_string(index=False))


Dernieres telemetries :
 pump_id                        timestamp  pressure  vibration   flow
pump_024 2026-06-19T18:31:01.480097+00:00     4.302     0.0440  9.827
pump_025 2026-06-19T18:31:01.480097+00:00     4.322     0.0437 10.062
pump_026 2026-06-19T18:31:01.480097+00:00     4.239     0.0502 10.219
pump_027 2026-06-19T18:31:01.480097+00:00     4.132     0.0463  9.996
pump_018 2026-06-19T18:31:01.478384+00:00     0.000     0.0769  0.000

Dernieres predictions :
 pump_id                        timestamp     prediction
pump_026 2026-06-19T18:31:04.162381+00:00 non functional
pump_025 2026-06-19T18:31:04.062249+00:00     functional
pump_024 2026-06-19T18:31:03.959259+00:00 non functional
pump_023 2026-06-19T18:31:03.868816+00:00 non functional
pump_022 2026-06-19T18:31:03.778951+00:00     functional


## 7. Latence MQTT-to-Prediction

D'après les logs du Consumer affichés dans la console, la latence typique d'une prédiction + écriture en base est sous les 5 ms.
Ceci remplit largement le critère d'acceptation de latence **< 5 secondes**.

## 8. Nettoyage (Arrêt des processus)

In [5]:
try:
    sim_proc.terminate()
    cons_proc.terminate()
    print("Processus arretes.")
except NameError:
    print("Processus deja arretes.")


Processus arretes.
